In [12]:
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
model = tf.keras.models.load_model("/content/drive/MyDrive/efficientnet_stage2.keras")


In [28]:
test_real = tf.data.Dataset.list_files(
    "/content/drive/MyDrive/face_detection_test/face_real/*.jpg",
    shuffle=False
)

test_fake = tf.data.Dataset.list_files(
    "/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",
    shuffle=False
)

In [29]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [30]:
def add_label(image, label):
  return image , label

test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [31]:
test_dataset = test_real.concatenate(test_fake)

In [32]:
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
test_dataset = test_dataset.map(preprocess)

In [33]:
BATCH_SIZE = 32
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [34]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 156s 2s/step - accuracy: 0.5790 - loss: 0.6714
Test Loss: 0.6713865995407104
Test Accuracy: 0.5789999961853027
